In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.metrics import make_scorer, recall_score
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

In [2]:
subset_a = pd.read_csv('/content/drive/MyDrive/L5 Sem2 ML/CW/loan_classification_data.csv')

In [3]:
subset_a.shape

(58645, 20)

In [4]:
subset_a.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58645 entries, 0 to 58644
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   age                            58645 non-null  float64
 1   income                         58645 non-null  float64
 2   employment_length              58645 non-null  float64
 3   loan_amount                    58645 non-null  float64
 4   loan_interest_rate             58645 non-null  float64
 5   loan_income_ratio              58645 non-null  float64
 6   credit_history_length          58645 non-null  float64
 7   home_ownership_MORTGAGE        58645 non-null  float64
 8   home_ownership_OTHER           58645 non-null  float64
 9   home_ownership_OWN             58645 non-null  float64
 10  home_ownership_RENT            58645 non-null  float64
 11  loan_intent_DEBTCONSOLIDATION  58645 non-null  float64
 12  loan_intent_EDUCATION          58645 non-null 

In [5]:
subset_a.head()

,age,income,employment_length,loan_amount,loan_interest_rate,loan_income_ratio,credit_history_length,home_ownership_MORTGAGE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,payment_default_on_file_N,payment_default_on_file_Y,loan_approval_status
0,-1.087927,-1.479895,-1.204656,1.039305,-1.214711,-0.427932,-0.450108,-0.856727,-0.038986,4.205788,-1.044345,-0.429488,1.944005,-0.346305,-0.478719,-0.453837,-0.4537,0.417363,-0.417363,0
1,-1.087927,-1.445688,-0.691706,2.836651,2.006927,0.335502,-0.698298,-0.856727,-0.038986,4.205788,-1.044345,-0.429488,1.944005,-0.346305,-0.478719,-0.453837,-0.4537,-2.395998,2.395998,0
2,-0.755699,-1.548309,0.077718,3.735324,0.573990,1.644245,-0.698298,-0.856727,-0.038986,-0.237768,0.957538,-0.429488,-0.514402,-0.346305,2.088910,-0.453837,-0.4537,0.417363,-0.417363,0
3,2.068236,3.366182,-0.435231,4.633996,-0.882006,0.335502,1.287227,-0.856727,-0.038986,-0.237768,0.957538,-0.429488,1.944005,-0.346305,-0.478719,-0.453837,-0.4537,0.417363,-0.417363,0
4,2.068236,0.743547,-0.435231,4.633996,0.573990,2.516741,2.031798,1.167232,-0.038986,-0.237768,-1.044345,-0.429488,-0.514402,2.887625,-0.478719,-0.453837,-0.4537,0.417363,-0.417363,0


In [6]:
X = subset_a.drop(columns=['loan_approval_status'])
y = subset_a['loan_approval_status']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train, 20% test
    random_state=42,    # ensures reproducibility
    stratify=y          # maintains class ratio in both splits
)

80:20 split
Class ratio is 85:15

In [8]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (46916, 19)
X_test shape: (11729, 19)
y_train shape: (46916,)
y_test shape: (11729,)


In [9]:
print("\nTraining set class ratio:\n", y_train.value_counts(normalize=True))
print("\nTest set class ratio:\n", y_test.value_counts(normalize=True))


Training set class ratio:
 loan_approval_status
0    0.857618
1    0.142382
Name: proportion, dtype: float64

Test set class ratio:
 loan_approval_status
0    0.857618
1    0.142382
Name: proportion, dtype: float64


The class ratio is maintained in both the training and test sets.

In [10]:
lr_model = LogisticRegression(random_state=42)
nb_model = GaussianNB()
knn_model = KNeighborsClassifier(n_neighbors=5)

Initialised the models

In [11]:
lr_model.fit(X_train, y_train)
nb_model.fit(X_train, y_train)
knn_model.fit(X_train, y_train)

KNeighborsClassifier()

Trained the model on the training data

In [12]:
lr_predictions = lr_model.predict(X_test)
nb_predictions = nb_model.predict(X_test)
knn_predictions = knn_model.predict(X_test)

In [13]:
lr_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear']
}

nb_params = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

knn_params = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

In [14]:
rejected_recall = make_scorer(recall_score, pos_label=1)

lr_grid = GridSearchCV(
    LogisticRegression(random_state=42, class_weight='balanced'),
    lr_params,
    cv=5,
    scoring=rejected_recall,
    n_jobs=-1
)

nb_grid = GridSearchCV(GaussianNB(), nb_params, cv=5, scoring=rejected_recall, n_jobs=-1)

knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5, scoring=rejected_recall, n_jobs=-1)

In [15]:
lr_grid.fit(X_train, y_train)

nb_grid.fit(X_train, y_train)
nb_probs = nb_grid.predict_proba(X_test)[:, 1]
nb_high_recall_preds = (nb_probs > 0.3).astype(int)

knn_grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'metric': ['euclidean', 'manhattan'],
                         'n_neighbors': [3, 5, 7, 9, 11],
                         'weights': ['uniform', 'distance']},
             scoring=make_scorer(recall_score, response_method='predict', pos_label=1))

In [16]:
print("Best LR parameters:", lr_grid.best_params_)
print("Best NB parameters:", nb_grid.best_params_)
print("Best KNN parameters:", knn_grid.best_params_)

Best LR parameters: {'C': 0.01, 'solver': 'liblinear'}
Best NB parameters: {'var_smoothing': 1e-09}
Best KNN parameters: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}


The best parameters for each model

In [17]:
lr_best_predictions = lr_grid.predict(X_test)
nb_best_predictions = nb_grid.predict(X_test)
knn_best_predictions = knn_grid.predict(X_test)

In [21]:
def evaluate_model(model_name, y_test, predictions, model, X_test):
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, predictions)
    cm_df = pd.DataFrame(
    cm,
    index=['Actual Approved', 'Actual Rejected'],
    columns=['Predicted Approved', 'Predicted Rejected']
    )

    fig_cm = px.imshow(
        cm_df,
        text_auto=True,
        color_continuous_scale='Blues',
        title=f'Confusion Matrix - {model_name}'
    )
    fig_cm.show()

    # Classification Report
    print(f"\nClassification Report - {model_name}")
    print(classification_report(y_test, predictions, target_names=['Approved', 'Rejected']))

    # AUC-ROC Curve
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    fig_roc = go.Figure()
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'ROC Curve (AUC = {roc_auc:.2f})'
    ))
    fig_roc.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        mode='lines',
        line=dict(dash='dash', color='gray'),
        name='Random Classifier'
    ))
    fig_roc.update_layout(
        title=f'AUC-ROC Curve - {model_name}',
        xaxis_title='False Positive Rate',
        yaxis_title='True Positive Rate'
    )
    fig_roc.show()

    print(f"AUC Score: {roc_auc:.2f}")

    return roc_auc

Reusable evaluation function

In [19]:
evaluate_model("Logistic Regression", y_test, lr_predictions, lr_model, X_test)
evaluate_model("Naive Bayes", y_test, nb_predictions, nb_model, X_test)
evaluate_model("KNN", y_test, knn_predictions, knn_model, X_test)


Model: Logistic Regression



Classification Report - Logistic Regression
              precision    recall  f1-score   support

    Approved       0.91      0.97      0.94     10059
    Rejected       0.73      0.43      0.54      1670

    accuracy                           0.90     11729
   macro avg       0.82      0.70      0.74     11729
weighted avg       0.89      0.90      0.88     11729



AUC Score: 0.89

Model: Naive Bayes



Classification Report - Naive Bayes
              precision    recall  f1-score   support

    Approved       0.95      0.84      0.89     10059
    Rejected       0.43      0.73      0.54      1670

    accuracy                           0.82     11729
   macro avg       0.69      0.79      0.71     11729
weighted avg       0.88      0.82      0.84     11729



AUC Score: 0.85

Model: KNN



Classification Report - KNN
              precision    recall  f1-score   support

    Approved       0.93      0.98      0.95     10059
    Rejected       0.81      0.56      0.66      1670

    accuracy                           0.92     11729
   macro avg       0.87      0.77      0.81     11729
weighted avg       0.91      0.92      0.91     11729



AUC Score: 0.87


np.float64(0.8650966483376819)

In [20]:
evaluate_model("Logistic Regression (Optimised)", y_test, lr_best_predictions, lr_grid, X_test)
evaluate_model("Naive Bayes (Optimised)", y_test, nb_high_recall_preds, nb_grid, X_test)
evaluate_model("KNN (Optimised)", y_test, knn_best_predictions, knn_grid, X_test)


Model: Logistic Regression (Optimised)



Classification Report - Logistic Regression (Optimised)
              precision    recall  f1-score   support

    Approved       0.96      0.79      0.87     10059
    Rejected       0.40      0.82      0.54      1670

    accuracy                           0.80     11729
   macro avg       0.68      0.81      0.70     11729
weighted avg       0.88      0.80      0.82     11729



AUC Score: 0.89

Model: Naive Bayes (Optimised)



Classification Report - Naive Bayes (Optimised)
              precision    recall  f1-score   support

    Approved       0.96      0.76      0.85     10059
    Rejected       0.36      0.80      0.50      1670

    accuracy                           0.77     11729
   macro avg       0.66      0.78      0.67     11729
weighted avg       0.87      0.77      0.80     11729



AUC Score: 0.85

Model: KNN (Optimised)



Classification Report - KNN (Optimised)
              precision    recall  f1-score   support

    Approved       0.93      0.97      0.95     10059
    Rejected       0.75      0.56      0.64      1670

    accuracy                           0.91     11729
   macro avg       0.84      0.77      0.80     11729
weighted avg       0.91      0.91      0.91     11729



AUC Score: 0.84


np.float64(0.8419221205665021)